In [3]:
from pathlib import Path
import pandas as pd
import os

path = os.path.join(os.getcwd(), "..", "..", "singapore_weather_monthly")

root = Path(path)


In [4]:
root

WindowsPath('c:/Computer Science/Hackathons/huawei/naisc-2025/model/flood/../../singapore_weather_monthly')

In [5]:

# the full schema you expect
EXPECTED_COLS = [
    "timestamp",
    "temperature",
    "rainfall",
    "humidity",
    "wind_direction",
    "wind_speed",
]

station_parts = {}

for month_dir in sorted(root.iterdir()):
    if not month_dir.is_dir():
        continue

    for csv_path in month_dir.glob("*.csv"):
        station = csv_path.stem

        # try reading with header; fall back to no-header
        try:
            df = pd.read_csv(csv_path, parse_dates=["timestamp"])
        except ValueError:
            df = pd.read_csv(
                csv_path,
                header=None,
                names=EXPECTED_COLS,
                parse_dates=[0],
            )

        # normalize column names
        df.columns = [c.lower() for c in df.columns]

        # ensure *all* expected columns are present, filling missing ones with <NA>
        df = df.reindex(
            columns=EXPECTED_COLS,
            fill_value=pd.NA,
        )

        # set timestamp index
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.set_index("timestamp").sort_index()

        station_parts.setdefault(station, []).append(df)

# concatenate per-station
for station, parts in station_parts.items():
    full = pd.concat(parts)
    full = full[~full.index.duplicated()]  # drop any true duplicates
    full = full.sort_index()

    # save or keep in memory
    full.to_csv(f"{station}_all_timeseries.csv")
    station_parts[station] = full


C:\Users\Parvez\AppData\Local\Temp\ipykernel_23340\1417172302.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(
C:\Users\Parvez\AppData\Local\Temp\ipykernel_23340\1417172302.py:41: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"])


DateParseError: Unknown datetime string format, unable to parse: S109, at position 1

In [6]:
import os
import glob
import pandas as pd

# 1. Define folder and file pattern
data_folder = os.path.join(os.getcwd(), "..", "..", "singapore_weather_combined") # adjust if needed
csv_pattern = os.path.join(data_folder, '*.csv')

# 2. Define full hourly datetime index
start = '2024-01-01 00:00:00'
end = '2025-04-26 23:00:00'
full_index = pd.date_range(start=start, end=end, freq='H')

# 3. Required columns
required_cols = ['timestamp', 'temperature', 'rainfall', 'humidity', 'wind_direction', 'wind_speed']

# 4. Process each CSV
for filepath in glob.glob(csv_pattern):
    filename = os.path.basename(filepath)
    # Skip metadata or non-station files if necessary
    if filename == 'combined_station_metadata.csv':
        continue

    # Read original CSV
    df = pd.read_csv(filepath, parse_dates=['timestamp'])

    # Ensure timestamp column exists
    if 'timestamp' not in df.columns:
        continue  # or handle appropriately

    # Set timestamp as index
    df = df.set_index('timestamp')

    # Reindex to full hourly range
    df = df.reindex(full_index)

    # Reset index to column
    df = df.reset_index().rename(columns={'index': 'timestamp'})

    # Ensure all required columns exist
    for col in required_cols:
        if col not in df.columns:
            df[col] = pd.NA

    # Reorder columns
    df = df[required_cols]

    # Save standardized CSV
    out_path = os.path.join(data_folder, 'standardized_' + filename)
    df.to_csv(out_path, index=False)

    print(f'Standardized {filename} -> standardized_{filename}')


C:\Users\Parvez\AppData\Local\Temp\ipykernel_23340\3862270781.py:12: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(start=start, end=end, freq='H')


Standardized S06_Paya_Lebar.csv -> standardized_S06_Paya_Lebar.csv
Standardized S07_Lornie_Road.csv -> standardized_S07_Lornie_Road.csv
Standardized S08_Upper_Thomson_Road.csv -> standardized_S08_Upper_Thomson_Road.csv
Standardized S102_Semakau_Landfill.csv -> standardized_S102_Semakau_Landfill.csv
Standardized S104_Woodlands_Avenue_9.csv -> standardized_S104_Woodlands_Avenue_9.csv
Standardized S106_Pulau_Ubin.csv -> standardized_S106_Pulau_Ubin.csv
Standardized S107_East_Coast_Parkway.csv -> standardized_S107_East_Coast_Parkway.csv
Standardized S108_Marina_Gardens_Drive.csv -> standardized_S108_Marina_Gardens_Drive.csv
Standardized S109_Ang_Mo_Kio_Avenue_5.csv -> standardized_S109_Ang_Mo_Kio_Avenue_5.csv
Standardized S111_Scotts_Road.csv -> standardized_S111_Scotts_Road.csv
Standardized S112_Lim_Chu_Kang_Road.csv -> standardized_S112_Lim_Chu_Kang_Road.csv
Standardized S113_Marine_Parade_Road.csv -> standardized_S113_Marine_Parade_Road.csv
Standardized S114_Choa_Chu_Kang_Avenue_4.csv -

In [ ]:
import os
import glob
import pandas as pd

# Folder containing standardized CSVs
data_folder = os.path.join(os.getcwd(), "..", "..", "singapore_weather_combined")
pattern = os.path.join(data_folder, 'standardized_*.csv')

# Process each standardized CSV
for filepath in glob.glob(pattern):
    filename = os.path.basename(filepath)
    print(f'Processing {filename}...')

    # Read CSV and parse timestamp
    df = pd.read_csv(filepath, parse_dates=['timestamp'])
    
    # Set timestamp as index
    df = df.set_index('timestamp').sort_index()
    
    # Perform linear interpolation for gaps <= 3 hours
    # limit=4 ensures only runs of NaNs of length up to 3 are filled
    df_interpolated = df.interpolate(
        method='time',
        limit=3,
        limit_direction='both'
    )
    
    # Write output
    out_path = os.path.join(data_folder, 'imputed_' + filename)
    df_interpolated.reset_index().to_csv(out_path, index=False)
    
    print(f'  Saved imputed data to {os.path.basename(out_path)}')
